# Faculty point 10: open-source alternatives to Jev, fine-tuned on DevMap

**Jev** (TypeSafe AI, Sept 2026) is a new *closed* classifier: you give it a text and a question with fixed options, it returns a probability per option. It cannot be fine-tuned. Its open-source alternatives can:

- **Laya** (Convai): ModernBERT-large, 421M parameters, Apache-2.0
- **Kev-0.8B** (Jared Palmer): Qwen3.5-0.8B + LoRA, Apache-2.0

Each is fine-tuned on DevMap **with its own official training code**, on the same folds as all our other models (new applications, 5 piles). Every row becomes the question *"Which device runs this OpenCL kernel faster for this job: CPU or GPU?"*, with the GPU name, data size, work-group size and kernel code as the text.

**How to run:**
1. **Runtime → Change runtime type → T4 GPU**, save.
2. **Runtime → Run all.** Allow Google Drive (tick **Select all**).
3. Order: **Laya** on AMD and NVIDIA first (about 1 hour), then **Kev** (slower, a few hours). Each finished pile is saved to `devmap_results` on your Drive at once.
4. **If you must stop** (or Colab disconnects): run all again later; finished piles are skipped.
5. At the end (or when you stop), run the last two cells and send Claude the screenshot and the downloaded `results_backup.zip`.

In [ ]:
import os, shutil, subprocess, sys, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - use Runtime > Change runtime type > T4 GPU")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=5.17,<6", "peft>=0.21", "accelerate",
                "datasets", "pydantic", "laya==0.4.15"], check=True)
if not os.path.exists("kev"):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "https://github.com/jaredpalmer/kev.git"], check=True)
if not os.path.exists("GM_SOURCE_CODE"):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "https://github.com/marco-siino/GM_SOURCE_CODE.git"], check=True)
os.makedirs("data", exist_ok=True)
shutil.copy("GM_SOURCE_CODE/AMD_DS/dataset-devmap-amd.csv", "data")
shutil.copy("GM_SOURCE_CODE/NVIDIA_DS/dataset-devmap-nvidia.csv", "data")
import transformers; print("transformers", transformers.__version__, "| Ready.")

## Helper code (no need to read or change)

In [ ]:
%%writefile devmap.py
"""
Shared helpers: load the DevMap dataset and build the four "exams".

Words used here (same as in our chats):
  row      = one line of the dataset (code + two job numbers + answer)
  program  = one piece of OpenCL code (a "kernel"); several rows can share it
  study    = the rows a model learns from   (training set)
  exam     = the rows it is scored on       (test set)

  application = the benchmark a program was taken from (e.g. "npb-3.3-BT");
                one application contains several programs. 71 in total.
  collection  = the benchmark suite an application belongs to (NPB, Rodinia,
                ...). 7 in total.

The four exams, from easiest to hardest:
  old             - the base paper's exam: shuffle the ROWS, cut into 5 piles.
                    Rows of the same program can land on both sides.
  unseen_program  - shuffle the PROGRAMS, cut into 5 piles. Every row of a
                    program is on one side only.
  unseen_app      - shuffle the APPLICATIONS, cut into 5 piles. Needed because
                    programs of one application share the same job numbers,
                    so those numbers give the application away.
  unseen_suite    - hold out one whole collection at a time, 7 rounds.
"""
import os
import re

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

DATA_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "data")
FILES = {"amd": "dataset-devmap-amd.csv", "nvidia": "dataset-devmap-nvidia.csv"}

# The hand-counted features that ship with the dataset (Grewe et al.):
#   comp      = number of compute operations in the code
#   rational  = number of relational (comparison) operations
#   mem       = number of memory accesses
#   localmem  = number of accesses to local memory
#   coalesced = number of memory accesses that are neatly side by side
#   atomic    = number of atomic operations (always 0 here)
#   transfer  = how much data is sent to the device  (job number 1)
#   wgsize    = how the work is batched               (job number 2)
STATIC_FEATURES = ["comp", "rational", "mem", "localmem", "coalesced", "atomic"]
JOB_FEATURES = ["transfer", "wgsize"]

SUITES = ["amd-sdk", "npb", "nvidia-sdk", "parboil", "polybench", "rodinia", "shoc"]


def suite_of(benchmark_name):
    """Which collection a program comes from, read off its name."""
    b = benchmark_name.lower()
    if b.startswith("amd"):
        return "amd-sdk"
    if b.startswith("npb"):
        return "npb"
    if b.startswith("nvidia"):
        return "nvidia-sdk"
    if b.startswith("parboil"):
        return "parboil"
    if b.startswith("polybench"):
        return "polybench"
    if b.startswith("rodinia"):
        return "rodinia"
    if b.startswith("shoc"):
        return "shoc"
    raise ValueError("unknown collection for " + benchmark_name)


def app_of(benchmark_name):
    """'npb-3.3-BT-exact_rhs1' -> 'npb-3.3-BT' (collection, version, application)."""
    m = re.match(r"^(.*?-\d+(?:\.\d+)*)-([^-]+)-(.*)$", benchmark_name)
    if not m:
        raise ValueError("cannot read application from " + benchmark_name)
    return m.group(1) + "-" + m.group(2)


def code_key(src):
    """Two rows are 'the same program' when their code text is identical
    once extra spaces and line breaks are ignored."""
    return re.sub(r"\s+", " ", str(src)).strip()


def load(machine, data_dir=DATA_DIR):
    """Load one answer sheet ('amd' or 'nvidia') with a few helper columns."""
    df = pd.read_csv(os.path.join(data_dir, FILES[machine]), sep=",")
    df = df.loc[:, [c for c in df.columns if not c.startswith("Unnamed")]].copy()
    df["row_id"] = np.arange(len(df))           # stable id = original file order
    df["suite"] = df["benchmark"].map(suite_of)
    df["app"] = df["benchmark"].map(app_of)
    keys = df["src"].map(code_key)
    df["program_id"] = pd.factorize(keys)[0]     # same code text -> same id
    df["label"] = (df["oracle"] == "GPU").astype(int)   # 0 = CPU, 1 = GPU
    return df


# ----------------------------------------------------------------------------
# The four exams. Each returns a list of rounds; a round is a dict with
# 'name', 'study' (row_ids) and 'exam' (row_ids).
# ----------------------------------------------------------------------------
def old_exam(df):
    """Exactly what the base paper's notebooks do:
    df.sample(frac=1, random_state=1) and then KFold(5, shuffle=False)."""
    shuffled = df.sample(frac=1, random_state=1).reset_index(drop=True)
    rounds = []
    for i, (tr, te) in enumerate(KFold(n_splits=5, shuffle=False).split(shuffled)):
        rounds.append({
            "name": f"pile{i + 1}",
            "study": shuffled.iloc[tr]["row_id"].tolist(),
            "exam": shuffled.iloc[te]["row_id"].tolist(),
        })
    return rounds


def group_piles(groups, labels, n_piles=5, seed=1):
    """Deal whole groups (programs or applications) into n_piles piles.

    Our own version instead of scikit-learn's StratifiedGroupKFold, because
    that one gives different piles in different library versions (Colab's
    1.6 put 364 rows in one AMD pile and 15 in another). This uses only
    numpy's fixed random generator, so the piles are the same everywhere.

    How it deals: shuffle the groups (seed), then take them biggest first and
    put each one in the pile where it keeps the piles most even, both in
    size and in their share of GPU answers. Returns one pile number per row."""
    groups = np.asarray(groups)
    labels = np.asarray(labels)
    names, inverse = np.unique(groups, return_inverse=True)
    sizes = np.bincount(inverse)
    gpus = np.bincount(inverse, weights=labels)
    order = np.random.RandomState(seed).permutation(len(names))
    order = order[np.argsort(-sizes[order], kind="stable")]       # biggest first, ties in shuffled order
    target_rows = len(groups) / n_piles
    target_gpu = max(labels.sum() / n_piles, 1.0)
    pile_rows = np.zeros(n_piles)
    pile_gpu = np.zeros(n_piles)
    pile_of_group = np.empty(len(names), dtype=int)
    for g in order:
        costs = []
        for p in range(n_piles):
            rows, gpu = pile_rows.copy(), pile_gpu.copy()
            rows[p] += sizes[g]
            gpu[p] += gpus[g]
            costs.append((((rows - target_rows) / target_rows) ** 2).sum()
                         + (((gpu - target_gpu) / target_gpu) ** 2).sum())
        best = int(np.argmin(costs))                               # ties -> lowest pile number
        pile_of_group[g] = best
        pile_rows[best] += sizes[g]
        pile_gpu[best] += gpus[g]
    return pile_of_group[inverse]


def fair_exam_unseen_programs(df, seed=1):
    """5 rounds; all rows of a program stay on the same side. Piles are also
    kept roughly balanced in size and CPU/GPU answers."""
    pile = group_piles(df["program_id"], df["label"], 5, seed)
    rounds = []
    for i in range(5):
        tr, te = np.where(pile != i)[0], np.where(pile == i)[0]
        rounds.append({
            "name": f"pile{i + 1}",
            "study": df.iloc[tr]["row_id"].tolist(),
            "exam": df.iloc[te]["row_id"].tolist(),
        })
    return rounds


def fair_exam_unseen_apps(df, seed=1):
    """5 rounds; all programs of an application stay on the same side. If the
    exact same code also exists under another application, those study rows
    are dropped so the exam code is truly unseen."""
    pile = group_piles(df["app"], df["label"], 5, seed)
    rounds = []
    for i in range(5):
        tr, te = np.where(pile != i)[0], np.where(pile == i)[0]
        exam = df.iloc[te]
        study = df.iloc[tr]
        study = study[~study["program_id"].isin(exam["program_id"])]
        rounds.append({
            "name": f"pile{i + 1}",
            "study": study["row_id"].tolist(),
            "exam": exam["row_id"].tolist(),
        })
    return rounds


def fair_exam_unseen_collection(df):
    """7 rounds; each round holds out one whole collection. If a held-out
    program's exact code also exists in another collection, those study rows
    are dropped so the exam code is truly unseen."""
    rounds = []
    for suite in SUITES:
        exam = df[df["suite"] == suite]
        study = df[(df["suite"] != suite) & (~df["program_id"].isin(exam["program_id"]))]
        rounds.append({
            "name": suite,
            "study": study["row_id"].tolist(),
            "exam": exam["row_id"].tolist(),
        })
    return rounds


def build_exam(df, exam, seed=1):
    if exam == "old":
        return old_exam(df)
    if exam == "unseen_program":
        return fair_exam_unseen_programs(df, seed)
    if exam == "unseen_app":
        return fair_exam_unseen_apps(df, seed)
    if exam == "unseen_suite":
        return fair_exam_unseen_collection(df)
    raise ValueError(exam)


EXAMS = ["old", "unseen_program", "unseen_app", "unseen_suite"]
RESHUFFLED_EXAMS = ["unseen_program", "unseen_app"]   # these change with the shuffle seed


def leak_report(df, rounds):
    """For each round: how many exam rows have code that is also in the
    study rows (seen) vs. not (new)."""
    pid = df.set_index("row_id")["program_id"]
    out = []
    for r in rounds:
        study_programs = set(pid.loc[r["study"]])
        seen = sum(1 for x in r["exam"] if pid.loc[x] in study_programs)
        out.append({"round": r["name"], "study_rows": len(r["study"]),
                    "exam_rows": len(r["exam"]), "exam_seen_code": seen,
                    "exam_new_code": len(r["exam"]) - seen})
    return pd.DataFrame(out)

In [ ]:
%%writefile 15_jev_alternatives.py
"""
Faculty point 10: open-source alternatives to Jev, fine-tuned on DevMap.

Jev (TypeSafe AI, Sept 2026) is a closed "System One" classifier: it reads a text
("state") plus a question with fixed options and returns a probability per option.
It cannot be fine-tuned. Its open-source alternatives can, and each is trained here
with ITS OWN official fine-tuning code, on the same folds as all our other models:

  laya   Convai's Laya: ModernBERT-large (421M) + decision head, Apache-2.0.
         Fine-tuned with laya.train.finetune (the package's own loop: RLCD loss,
         calibration on a 10% slice of the training rows).
  kev    Jared Palmer's Kev-0.8B: Qwen3.5-0.8B-Base + LoRA + pointer head, Apache-2.0.
         Fine-tuned with `python -m kev.train --init_from jaredpalmer/kev-0.8b`
         (the repo's documented "delta" fine-tune), then scored with kev's loader.

Every row becomes the same typed question:
  state    = which GPU, data copied, work-group size, then the kernel code
  question = "Which device runs this OpenCL kernel faster for this job?"  options CPU / GPU
Exam: unseen_app (shuffle seed 1), 5 piles, so the results line up row by row with
graphcodebert+fast, graphcodebert+e2e, the hybrid and the Random Forest.
Results: preds__<laya|kev>__<machine>__unseen_app__split1__train<seed>.csv, one
file per model/machine/seed, saved after every pile; finished piles are skipped.
Per-pile model checkpoints are written to local disk only and deleted after use.
"""
import json
import os
import shutil
import subprocess
import sys
import time

import numpy as np
import pandas as pd

import devmap

EXAM = "unseen_app"
GPU_NAME = {"amd": "AMD Radeon HD 7970", "nvidia": "NVIDIA GeForce GTX 970"}
QID = "device"
QUESTION = {
    "type": "choice",
    "instructions": "Which device runs this OpenCL kernel faster for this job: the CPU or the GPU?",
    "criteria": {"CPU": "the Intel Core i7-3820 CPU is faster", "GPU": "the GPU is faster"},
}


def make_state(row, machine):
    code = " ".join(str(row["src"]).split())
    return (f"Hardware: Intel Core i7-3820 CPU versus {GPU_NAME[machine]} GPU. "
            f"Data copied between CPU and GPU: {int(row['transfer'])} bytes. "
            f"Work-group size: {int(row['wgsize'])}. OpenCL kernel: {code}")


def label_name(y):
    return "GPU" if int(y) == 1 else "CPU"


# ----------------------------------------------------------------------------------- Laya

class LayaRunner:
    label = "laya"

    def __init__(self, base="convaiinnovations/laya", epochs=4, micro_batch=8, grad_accum=1, max_len=512,
                 device="auto", log=print):
        self.base, self.epochs, self.micro_batch, self.grad_accum = base, epochs, micro_batch, grad_accum
        self.max_len, self.device, self.log = max_len, device, log

    def fit_predict(self, study, exam, machine, seed, workdir):
        import laya
        from laya.train import TrainConfig, finetune
        os.makedirs(workdir, exist_ok=True)
        data = os.path.join(workdir, "train.jsonl")
        with open(data, "w") as f:
            for _, r in study.iterrows():
                f.write(json.dumps({"state": make_state(r, machine), "questions": {QID: QUESTION},
                                    "expected": {QID: label_name(r["label"])}}) + "\n")
        out = os.path.join(workdir, "model")
        cfg = TrainConfig(epochs=self.epochs, micro_batch=self.micro_batch, grad_accum=self.grad_accum,
                          seed=seed, max_len=self.max_len, log_every=50)
        finetune(data, self.base, out, cfg, device=self.device)
        agent = laya.load(out, device=None if self.device == "auto" else self.device)
        states = [make_state(r, machine) for _, r in exam.iterrows()]
        results = agent.predict_batch(states, {QID: QUESTION}, batch_size=16)
        del agent
        return np.array([gpu_probability(res) for res in results])


def gpu_probability(result):
    """P(GPU) from a Laya/Jev-style answer: the option probabilities if given, else the chosen option."""
    ans = result["answers"][QID]
    probs = ans.get("probabilities") or {}
    if "GPU" in probs or "CPU" in probs:
        g, c = float(probs.get("GPU", 0.0)), float(probs.get("CPU", 0.0))
        return g / (g + c) if g + c > 0 else 0.5
    return 1.0 if ans.get("choice") == "GPU" else 0.0


# ------------------------------------------------------------------------------------ Kev

class KevRunner:
    label = "kev"

    def __init__(self, kev_dir="kev", base="Qwen/Qwen3.5-0.8B-Base", init_from="jaredpalmer/kev-0.8b",
                 epochs=2, lr=2e-5, batch=1, accum=8, max_state=1024, dtype="fp32", device="auto",
                 extra_args=(), log=print):
        self.kev_dir, self.base, self.init_from = os.path.abspath(kev_dir), base, init_from
        self.epochs, self.lr, self.batch, self.accum = epochs, lr, batch, accum
        self.max_state, self.dtype, self.device, self.extra_args, self.log = max_state, dtype, device, list(extra_args), log
        self._tok = None

    def _env(self):
        env = dict(os.environ)
        env["PYTHONPATH"] = self.kev_dir + os.pathsep + env.get("PYTHONPATH", "")
        return env

    def _kev(self):
        if self.kev_dir not in sys.path:
            sys.path.insert(0, self.kev_dir)
        from kev import checkpoint, data, model
        return checkpoint, data, model

    def _states(self, df, machine):
        """States cut so the whole state fits the training context (kev drops, never truncates, longer records)."""
        _, _, kmodel = self._kev()
        if self._tok is None:
            self._tok = kmodel.load_tokenizer(self.base)
        budget = self.max_state - 64          # room for kev's delimiters and the state token
        out = []
        for _, r in df.iterrows():
            s = make_state(r, machine)
            ids = self._tok(s, add_special_tokens=False)["input_ids"]
            out.append(s if len(ids) <= budget else self._tok.decode(ids[:budget]))
        return out

    def fit_predict(self, study, exam, machine, seed, workdir):
        os.makedirs(workdir, exist_ok=True)
        data = os.path.join(workdir, "train.jsonl")
        with open(data, "w") as f:
            for s, (_, r) in zip(self._states(study, machine), study.iterrows()):
                f.write(json.dumps({"state": s, "questions": {QID: {**QUESTION, "label": label_name(r["label"])}}}) + "\n")
        out = os.path.join(workdir, "run")
        cmd = [sys.executable, "-m", "kev.train", "--data", data, "--base", self.base,
               "--epochs", str(self.epochs), "--lr", str(self.lr), "--batch", str(self.batch), "--accum", str(self.accum),
               "--dtype", self.dtype, "--device", self.device, "--max_state", str(self.max_state),
               "--p_none", "0", "--p_none_distract", "0", "--p_distract", "0",
               "--seed", str(seed), "--out", out] + (["--init_from", self.init_from] if self.init_from else []) + self.extra_args
        self.log("      " + " ".join(cmd[1:]))
        proc = subprocess.Popen(cmd, cwd=self.kev_dir, env=self._env(), stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        tail = []
        for line in proc.stdout:
            line = line.rstrip()
            tail = (tail + [line])[-40:]
            if any(k in line for k in ("epoch", "step", "dropped", "training requests", "delta:", "device=", "Error", "error")):
                self.log("      " + line[:200])
        if proc.wait() != 0:
            raise RuntimeError("kev.train failed:\n" + "\n".join(tail))
        return self.predict(out, exam, machine)

    def predict(self, run_dir, exam, machine):
        import torch
        kcheck, kdata, kmodel = self._kev()
        dev = torch.device("cuda" if torch.cuda.is_available() and self.device != "cpu" else "cpu")
        tok, m = kcheck.Checkpoint(run_dir).load(dev, kcheck.LoadOptions())
        ctx = kmodel.training_context(self.max_state)
        probs = []
        with torch.no_grad():
            for s in self._states(exam, machine):
                req = {"state": s, "questions": {QID: {**QUESTION, "label": "CPU", "src": "devmap_choice"}}}
                rec = kdata.materialize(req)
                p = m.probs(m.encode(tok, rec, max_state=ctx["max_state"], max_branch=ctx["max_branch"]))[0]
                keys = rec["questions"][0]["keys"]
                probs.append(float(p[keys.index("GPU")]))
        del m
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return np.array(probs)


# ------------------------------------------------------------------------------------ run

def result_path(out_dir, label, machine, seed):
    return os.path.join(out_dir, f"preds__{label}__{machine}__{EXAM}__split1__train{seed}.csv")


def run(runners, machines=("amd", "nvidia"), seeds=(1,), data_dir=devmap.DATA_DIR, out_dir="results",
        work_root="jev_work", log=print):
    os.makedirs(out_dir, exist_ok=True)
    for seed in seeds:
        for runner in runners:
            for machine in machines:
                df = devmap.load(machine, data_dir).set_index("row_id", drop=False)
                rounds = devmap.build_exam(df, EXAM, seed=1)
                path = result_path(out_dir, runner.label, machine, seed)
                done = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()
                log(f"\n=== {runner.label} | {machine} | exam '{EXAM}' | training seed {seed} ===")
                for r in rounds:
                    if len(done) and set(done.loc[done["round"] == r["name"], "row_id"]) == set(r["exam"]):
                        log(f"   round {r['name']}: already done, skipping")
                        continue
                    if len(done):
                        done = done[done["round"] != r["name"]]
                    study, ex = df.loc[r["study"]], df.loc[r["exam"]]
                    log(f"   round {r['name']}: study {len(study)} rows, exam {len(ex)} rows")
                    work = os.path.join(work_root, f"{runner.label}_{machine}_{seed}_{r['name']}")
                    t0 = time.time()
                    try:
                        prob = runner.fit_predict(study, ex, machine, seed, work)
                    finally:
                        shutil.rmtree(work, ignore_errors=True)
                    part = pd.DataFrame({
                        "model": runner.label, "machine": machine, "exam": EXAM, "split_seed": 1, "train_seed": seed,
                        "round": r["name"], "row_id": ex["row_id"].to_numpy(), "benchmark": ex["benchmark"].to_numpy(),
                        "suite": ex["suite"].to_numpy(), "app": ex["app"].to_numpy(),
                        "program_id": ex["program_id"].to_numpy(),
                        "code_seen_in_study": ex["program_id"].isin(set(study["program_id"])).to_numpy(),
                        "transfer": ex["transfer"].to_numpy(), "wgsize": ex["wgsize"].to_numpy(),
                        "label": ex["label"].to_numpy(), "pred": (prob >= 0.5).astype(int), "prob_gpu": prob})
                    done = pd.concat([done, part], ignore_index=True)
                    done.to_csv(path, index=False)
                    log(f"   round {r['name']}: exam accuracy {100 * (part.label == part.pred).mean():.1f}%  "
                        f"({(time.time() - t0) / 60:.1f} min)")
                log(f"   overall accuracy {100 * (done.label == done.pred).mean():.1f}%")


def summary(out_dir, labels=("laya", "kev", "graphcodebert+fast", "graphcodebert+e2e", "codeberta+fast")):
    import glob
    rows = []
    for f in glob.glob(os.path.join(out_dir, f"preds__*__{EXAM}__split1__train*.csv")):
        d = pd.read_csv(f)
        if len(d) == 680 and d["model"].iloc[0] in labels:
            rows.append({"model": d["model"].iloc[0], "machine": d["machine"].iloc[0],
                         "seed": d["train_seed"].iloc[0], "acc": 100 * (d.label == d.pred).mean()})
    if not rows:
        return pd.DataFrame()
    t = pd.DataFrame(rows).groupby(["model", "machine"]).acc.agg(["mean", "count"]).round(1).unstack("machine")
    t.columns = [f"{m} {'accuracy' if s == 'mean' else 'runs'}" for s, m in t.columns]
    return t.reset_index()

## Settings

In [ ]:
import os
MODELS = ["laya", "kev"]        # Laya first (faster), then Kev
MACHINES = ["amd", "nvidia"]
SEEDS = [1]                     # one training run first; add 2, 3 later for repeats

RESULTS_DIR = "results"
try:
    from google.colab import drive
    drive.mount("/content/drive")                      # tick "Select all"
    RESULTS_DIR = "/content/drive/MyDrive/devmap_results"
except ImportError:
    pass
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Saving to:", RESULTS_DIR)

## Fine-tune and sit the exam (resumable)

In [ ]:
import importlib, os
os.environ["USE_TF"] = "0"          # Laya's tip: avoids a TensorFlow hang while loading
J = importlib.import_module("15_jev_alternatives")
make = {"laya": lambda: J.LayaRunner(), "kev": lambda: J.KevRunner(kev_dir="kev")}
for name in MODELS:                  # one model at a time, both machines
    J.run([make[name]()], machines=MACHINES, seeds=SEEDS, data_dir="data", out_dir=RESULTS_DIR)
print("\nAll requested runs are finished.")

## Results so far (send Claude a screenshot)

In [ ]:
import importlib, pandas as pd
J = importlib.import_module("15_jev_alternatives")
pd.set_option("display.width", 200)
print(J.summary(RESULTS_DIR).to_string(index=False))
print("\nFor comparison: hybrid (GraphCodeBERT + Random Forest) 74.4 / 73.7 on these same piles (AMD / NVIDIA)")

## Download the results to your laptop too (a backup in case Drive fails)

In [ ]:
import glob, os, shutil
from google.colab import files
os.makedirs("backup", exist_ok=True)
for pattern in ["preds__laya__*", "preds__kev__*"]:
    for f in glob.glob(os.path.join(RESULTS_DIR, pattern)):
        shutil.copy(f, "backup")
shutil.make_archive("results_backup", "zip", "backup")
print("Files in backup:", sorted(os.listdir("backup")))
files.download("results_backup.zip")